# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="./assets/LC_Memory_before.png" style="width:500px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

## 环境准备

从仓库根目录加载并检查 `.env` 与依赖配置。

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# ルートの .env を読み込む
load_dotenv()

# ルート設定を確認する
doublecheck_env("example.env")

OLLAMA_BASE_URL=<not set>
OLLAMA_MODEL=<not set>


In [2]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [3]:
from dataclasses import dataclass


@dataclass
class RuntimeContext:
    db: SQLDatabase

In [4]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime


@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [5]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [6]:
import os

from langchain.agents import create_agent

agent = create_agent(
    model=f"ollama:{os.environ['OLLAMA_MODEL']}",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

## Repeated Queries

In [7]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?


================================== Ai Message ==================================

To find out the total amount of your last invoice, we need to identify which table contains the invoices and what columns are relevant. Let's start by checking the structure of the `invoices` table.
Tool Calls:
  execute_sql (15b0b8a0-8e1f-46d6-8e25-8c1cdc12b06f)
 Call ID: 15b0b8a0-8e1f-46d6-8e25-8c1cdc12b06f
  Args:
    query: SELECT column_name FROM invoices WHERE customer_id = 1 LIMIT 1
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: invoices
[SQL: SELECT column_name FROM invoices WHERE customer_id = 1 LIMIT 1]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


================================== Ai Message ==================================

It appears there is no `invoices` table in the database. Let's try a different approach by checking for a `sales_invoices` or similar table that might contain your invoice details.

I'll execute a query to list tables and see if we can find an appropriate one.
Tool Calls:
  execute_sql (efa1e4b3-7d98-4800-8342-f8a29dde1bb9)
 Call ID: efa1e4b3-7d98-4800-8342-f8a29dde1bb9
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' LIMIT 5
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',)]


================================== Ai Message ==================================

It seems there are no tables related to invoices or sales in the database. However, let's check if we can find any reference to your customer ID in the `Customer` table.

I'll run a query to see if there is an entry for you.
Tool Calls:
  execute_sql (ea589724-1ab3-4aaa-a6d6-98c0b94f68fa)
 Call ID: ea589724-1ab3-4aaa-a6d6-98c0b94f68fa
  Args:
    query: SELECT customer_id FROM Customer WHERE name = 'Frank Harris'
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such column: customer_id
[SQL: SELECT customer_id FROM Customer WHERE name = 'Frank Harris']
(Background on this error at: https://sqlalche.me/e/20/e3q8)


================================== Ai Message ==================================

It seems there is no `customer_id` column in the `Customer` table, but we can still look up your details by name. Let's proceed with that.

I will check if the customer 'Frank Harris' exists in the database.
Tool Calls:
  execute_sql (c8dac62b-76a6-44f2-915b-3865b460ebaa)
 Call ID: c8dac62b-76a6-44f2-915b-3865b460ebaa
  Args:
    query: SELECT id FROM Customer WHERE name = 'Frank Harris'
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such column: id
[SQL: SELECT id FROM Customer WHERE name = 'Frank Harris']
(Background on this error at: https://sqlalche.me/e/20/e3q8)


================================== Ai Message ==================================

It seems there is an issue with the query as it's trying to select a column `id` that might not exist. Let's try again by directly querying for the customer 'Frank Harris' and see if we can get any relevant information.

I will run the following query:
```sql
SELECT * FROM Customer WHERE name = 'Frank Harris';
```
Let's execute this.
Tool Calls:
  execute_sql (c8965108-44ac-4789-a67c-7f5dba0754e1)
 Call ID: c8965108-44ac-4789-a67c-7f5dba0754e1
  Args:
    query: SELECT * FROM Customer WHERE name = 'Frank Harris'
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such column: name
[SQL: SELECT * FROM Customer WHERE name = 'Frank Harris']
(Background on this error at: https://sqlalche.me/e/20/e3q8)


================================== Ai Message ==================================

It appears that the `Customer` table does not have a `name` column either. This suggests there might be a misunderstanding or missing information about how customer data is structured in this database.

Given the current structure, we cannot directly find Frank Harris's invoice details as the necessary columns do not exist in the tables we've checked so far.

Would you like to provide more details on the table and column names that should be used for customers and invoices? Alternatively, if there are other relevant tables or structures, please let me know.


In [8]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?


================================== Ai Message ==================================

To provide you with the titles, I need to know from which table or set of data you're referring. Could you please specify the table name or context? If you'd like information from a specific table, let me know the name and any other details necessary to construct an appropriate SQL query.


## Add memory

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

In [10]:
import os

from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    model=f"ollama:{os.environ['OLLAMA_MODEL']}",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

In [11]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?


================================== Ai Message ==================================

To find out the total of your last invoice, we first need to identify which invoice is considered "last" for you. Typically, this would be the most recent one based on the date it was issued.

Let's start by finding the latest invoice date.
Tool Calls:
  execute_sql (1e8f8c35-5cff-48f3-98cb-16312db92e15)
 Call ID: 1e8f8c35-5cff-48f3-98cb-16312db92e15
  Args:
    query: SELECT MAX(date) AS last_invoice_date FROM invoices WHERE customer_id = 1;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: invoices
[SQL: SELECT MAX(date) AS last_invoice_date FROM invoices WHERE customer_id = 1;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


================================== Ai Message ==================================

It seems there might be an issue with the table name or possibly the structure of the `invoices` table. Let's double-check by listing the tables in the database to ensure that the `invoices` table exists and contains a relevant column for identifying invoices, such as `customer_id` and `date`.

I will execute the following query:
```sql
SELECT name FROM sqlite_master WHERE type='table';
```
This will list all the tables in the current database.
Tool Calls:
  execute_sql (eb9867e6-9d24-419c-8231-be4107485fc9)
 Call ID: eb9867e6-9d24-419c-8231-be4107485fc9
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table'
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]


================================== Ai Message ==================================

The `invoices` table seems to be listed as `Invoice` in the database. Let's proceed with querying the latest invoice for Frank Harris.

We will execute the following query:
```sql
SELECT MAX(date) AS last_invoice_date FROM Invoice WHERE customer_id = 1;
```
This query should give us the date of the most recent invoice.
Tool Calls:
  execute_sql (f9a8d4fa-8932-4ace-90bc-44367fc9ec3e)
 Call ID: f9a8d4fa-8932-4ace-90bc-44367fc9ec3e
  Args:
    query: SELECT MAX(date) AS last_invoice_date FROM Invoice WHERE customer_id = 1;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such column: date
[SQL: SELECT MAX(date) AS last_invoice_date FROM Invoice WHERE customer_id = 1;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


================================== Ai Message ==================================

It appears that the `Invoice` table does not have a `date` column. Let's try to find out when Frank Harris' latest invoice was issued by checking other relevant columns, such as an `invoice_date`.

Let's execute the following query to get the latest invoice date for customer_id 1 (assuming this is Frank Harris):
```sql
SELECT MAX(invoice_date) AS last_invoice_date FROM Invoice WHERE customer_id = 1;
```
If this doesn't work, we will need to adjust our approach based on the actual schema.
Tool Calls:
  execute_sql (c1eb6263-7500-4624-bc2f-88d0a36998f8)
 Call ID: c1eb6263-7500-4624-bc2f-88d0a36998f8
  Args:
    query: SELECT MAX(invoice_date) AS last_invoice_date FROM Invoice WHERE customer_id = 1;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such column: invoice_date
[SQL: SELECT MAX(invoice_date) AS last_invoice_da

================================== Ai Message ==================================

It seems that neither `date` nor `invoice_date` columns exist in the `Invoice` table. Let's try another approach by checking if there is a column named something similar, such as `created_at` or `issued_date`.

We will execute the following query to see if any relevant column exists:
```sql
SELECT DISTINCT column_name FROM pragma_table_info('Invoice');
```
This query retrieves all distinct column names from the `Invoice` table. We can then identify which one might be used for tracking when an invoice was created or issued.


In [12]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?


================================== Ai Message ==================================

It seems we need to find out what columns are available in the `Invoice` table before proceeding with identifying the correct column name for the invoice date.

Let's first get a list of all column names from the `Invoice` table:
```sql
SELECT DISTINCT column_name FROM pragma_table_info('Invoice');
```
This will help us identify which column might be used to determine the latest invoice.
Tool Calls:
  execute_sql (6b2bca68-6faf-42ae-84f6-79cc03784081)
 Call ID: 6b2bca68-6faf-42ae-84f6-79cc03784081
  Args:
    query: SELECT DISTINCT column_name FROM pragma_table_info('Invoice');
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such column: column_name
[SQL: SELECT DISTINCT column_name FROM pragma_table_info('Invoice');]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


================================== Ai Message ==================================

It appears that there was an issue with the previous query. Let's try a different approach to get the list of columns from the `Invoice` table.

We will execute the following query:
```sql
SELECT name FROM pragma_table_info('Invoice');
```
This should give us a list of all column names in the `Invoice` table.


## Try your own queries
Now that there is memory, check the agents recall!

In [13]:
question = "Your Question Here?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

Your Question Here?


================================== Ai Message ==================================

Sure, let's try executing the query to get the list of columns from the `Invoice` table:
```sql
SELECT name FROM pragma_table_info('Invoice');
```
This will help us identify which columns are available in the `Invoice` table.
Tool Calls:
  execute_sql (1df93f28-9fe9-4a06-b592-4e2757728f99)
 Call ID: 1df93f28-9fe9-4a06-b592-4e2757728f99
  Args:
    query: SELECT name FROM pragma_table_info('Invoice');
================================= Tool Message =================================
Name: execute_sql

[('InvoiceId',), ('CustomerId',), ('InvoiceDate',), ('BillingAddress',), ('BillingCity',), ('BillingState',), ('BillingCountry',), ('BillingPostalCode',), ('Total',)]


================================== Ai Message ==================================

The `Invoice` table has the following columns:
- InvoiceId
- CustomerId
- InvoiceDate
- BillingAddress
- BillingCity
- BillingState
- BillingCountry
- BillingPostalCode
- Total

From this list, it looks like the `InvoiceDate` column is likely the one used to track when an invoice was created or issued. Let's use this column to find the latest invoice for Frank Harris.

We will execute the following query:
```sql
SELECT MAX(InvoiceDate) AS last_invoice_date FROM Invoice WHERE CustomerId = 1;
```
This should give us the date of the most recent invoice for customer_id 1 (Frank Harris).
